# Exhaustion Reversal

Strategy Exhaustion Reversal detects push → stall → opposite-trigger streak patterns via an incremental, no-look-ahead tracker.

__How Exhaustion Reversal Algorithm Determines Entry/Exit:__
- Enters at the trigger-bar close once per-candle volume exceeds the push leg's.
- Exits on intrabar stop (push/stall extreme ± ATR buffer), fixed R:R target, time stop, or a structural-invalidation streak.

## Configuration: automatic

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import ExhaustionParams
from engine.visualization import build_chart

In [ ]:
# Automatic config: canonical handles from the three configurators (project-wide defaults).
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
import dataclasses
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import ExhaustionParams, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE

DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = ExhaustionParams()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

## Configuration: manual

Per-notebook overrides on top of the automatic config above. Each cell applies
`dataclasses.replace` to one handle; **leave a dict empty (or `EXIT_POLICY = None`)
to keep that dimension automatic**. Everything below this chapter uses only
`df`, `SYMBOL`, `INTERVAL`, `STRATEGY_CONFIG`, `EXIT_POLICY`, `TRADING_CONFIG`.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic ExhaustionParams().
STRATEGY_OVERRIDES = {}      # e.g. {"exhaustion_push_min_len": 3, "exhaustion_target_rr": 2.5}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

In [ ]:
# Resolve the final inputs the rest of the notebook uses. (Runs after overrides.)
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

## Exhaustion Reversal

In [5]:
# Import Exhaustion Reversal strategy
from engine.strategies import ExhaustionReversalStrategy


In [ ]:
# Backtest Exhaustion Reversal strategy
strategy = ExhaustionReversalStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [7]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")


─────────────────────────────────────────────
  Dollar P&L (starting $100.00)
─────────────────────────────────────────────
  #  1  short     -0.68  →  $99.32
  #  2  short     +0.78  →  $100.10
  #  3  short     -0.45  →  $99.64
  #  4  long      -0.50  →  $99.15
  #  5  long      +0.48  →  $99.63
  #  6  short     -0.61  →  $99.02
  #  7  long      -0.56  →  $98.46
  #  8  short     +0.42  →  $98.88
  #  9  long      -0.53  →  $98.35
  # 10  short     -1.09  →  $97.26
  # 11  long      -0.65  →  $96.61
  # 12  short     +0.02  →  $96.63
  # 13  short     -0.75  →  $95.88
  # 14  long      -0.46  →  $95.42
  # 15  long      -0.54  →  $94.88
  # 16  short     -0.75  →  $94.13
  # 17  short     -0.33  →  $93.80
  # 18  long      -0.60  →  $93.19
  # 19  long      +0.98  →  $94.17
  # 20  short     -0.28  →  $93.89
  # 21  long      -0.02  →  $93.87
  # 22  short     -0.22  →  $93.65
  # 23  short     -0.12  →  $93.53
  # 24  long      +0.00  →  $93.53
  # 25  long      -0.03  →  $93.49

In [ ]:
# Exhaustion Reversal strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Live signals

Live mode:
- It generates signals, it does not place orders. \
There's no exchange API key, no order execution. It tells you when to enter/exit.
- State persists \
If you stop and restart, it remembers whether you're in a position via its SQLite state file under data/live/.
- Circuit breaker \
If Bybit is unreachable 10 times in a row, it stops automatically instead of spinning forever.
- Chart updates in place \
Automatic: the chart refreshes in the browser every poll_seconds.

To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.

If you don't want to re-download automatically, two edits locally:
1. visualization.py — add auto_refresh: int = 0 parameter to build_chart, and after fig.write_html(save_path):
pythonif auto_refresh > 0:
    with open(save_path, "r") as f:
        html = f.read()
    meta_tag = f'<meta http-equiv="refresh" content="{auto_refresh}">'
    html = html.replace("<head>", f"<head>{meta_tag}", 1)
    with open(save_path, "w") as f:
        f.write(html)
2. live.py — add auto_refresh=self.poll_seconds to the build_chart() call in _tick().

### From CLI (command line interface)

- runs in a loop
- polls (refreshes) Bybit every 30 seconds
- persists state to SQLite (survives restarts)
- writes a chart under data/live/ each tick
- handles SIGTERM/Ctrl+C gracefully

In [ ]:
python -m engine \
    --strategy exhaustion_reversal \
    --mode live \
    --symbol BTCUSDT \
    --interval 15 \
    --candles 500 \
    --poll 30

### From a notebook cell

In [ ]:
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategy_configurator import ExhaustionParams
from engine.strategies import ExhaustionReversalStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"

config = ExhaustionParams()
strategy = ExhaustionReversalStrategy(config)

engine = LiveEngine(
    strategy=strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=30,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{strategy.name}.db"),
)

engine.run()  # blocks until Ctrl+C or kernel interrupt